# S2 α-sweep with a Vision Transformer (ViT) backbone

Reviewer-motivated backbone-sensitivity validation for the Sentinel-2 case study. Alongside the XGBoost baseline in `s2_alpha_sweep.ipynb`, we rerun the 29 (tile, scale) configurations with a **Vision Transformer** adapted for multispectral 5\,m Sentinel-2 patches.

**Architecture** (inspired by~\citet{dosovitskiy2021vit}, adapted to 13-band S2 input):

- Input: $16 \times 16 \times 13$ spatial patch centered on each labeled pixel.
- Split into $4 \times 4$ sub-patches $\to$ 16 spatial tokens, each flattened to $16 \times 13 = 208$-dim.
- Linear embed $\to$ dim 96. Prepend a CLS token. Add learnable positional embedding (17 positions).
- 6-layer Transformer encoder, 8 heads, FFN dim 256, pre-LayerNorm, GELU, dropout 0.1.
- CLS output $\to$ per-class classifier head (classifier head is adapted to each patch's remaining class count $K$).

**Protocol** matches `s2_alpha_sweep.ipynb`: same 10 tiles, same 3 scales (1, 2, 5\,km), same 60/20/20 stratified split, same joint $(r, \text{bw})$ CV, same $\alpha$ grid.

The key setting difference from XGBoost: the ViT sees $16 \times 16$ spatial context around each labeled pixel. Train pixels' patches overlap calibration/test pixels' features, pushing the S2 case from **inductive** (XGBoost, per-pixel) to **transductive** (ViT, patch-based). Coverage is then guaranteed via the permutation-invariance argument of~\citet{liu2024sacp} (see~\S3 of the main paper). This is intentional: combined with S2-XGBoost results, we demonstrate that GeoCP-RS is valid and effective in both exchangeability regimes on both modalities.

Outputs: `/content/drive/MyDrive/s2_vit_alpha_sweep/checkpoints/{tile}_s{size}.pkl`.

**Runtime**: $\sim$3--5 hours on a Colab T4 (30 configs $\times$ ~7 min each).

## 1. Install + GPU check

In [3]:
!pip install -q scikit-learn scipy pandas
import os, sys, time, gc, pickle, json, shutil
import numpy as np
import torch
print('python', sys.version.split()[0], '| torch', torch.__version__, '| cuda', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))
!nvidia-smi 2>/dev/null | head -10

python 3.12.13 | torch 2.10.0+cu128 | cuda True
GPU: Tesla T4
Sun Apr 19 16:57:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |


## 2. Mount Drive + paths

In [4]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

WORK_DIR   = '/content/drive/MyDrive/sentinel2_landcover_pilot_10m'
TILE_DIR   = f'{WORK_DIR}/tiles'

LOCAL_DIR   = '/content/s2_vit_alpha_sweep'
CKPT_DIR    = f'{LOCAL_DIR}/checkpoints'
RESULTS_DIR = f'{LOCAL_DIR}/results'
for d in [LOCAL_DIR, CKPT_DIR, RESULTS_DIR]: os.makedirs(d, exist_ok=True)

DRIVE_MIRROR   = '/content/drive/MyDrive/s2_vit_alpha_sweep'
DRIVE_CKPT_DIR = f'{DRIVE_MIRROR}/checkpoints'
try: os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
except OSError: pass

def _restore():
    if not os.path.isdir(DRIVE_CKPT_DIR): return 0
    try: pkls = [f for f in os.listdir(DRIVE_CKPT_DIR) if f.endswith('.pkl')]
    except OSError: return 0
    n = 0
    for f in pkls:
        src, dst = f'{DRIVE_CKPT_DIR}/{f}', f'{CKPT_DIR}/{f}'
        if os.path.exists(dst): continue
        try: shutil.copy2(src, dst); n += 1
        except OSError as e: print(f'[restore] {f}: {e}')
    return n
print(f'[restore] {_restore()} cached pkls from Drive')

tiles_present = sorted(f.replace('.npz','') for f in os.listdir(TILE_DIR) if f.endswith('.npz'))
print(f'TILE_DIR : {TILE_DIR}  ({len(tiles_present)} tiles)')
print(f'CKPT_DIR : {CKPT_DIR}  (local primary)')

Mounted at /content/drive
[restore] 2 cached pkls from Drive
TILE_DIR : /content/drive/MyDrive/sentinel2_landcover_pilot_10m/tiles  (10 tiles)
CKPT_DIR : /content/s2_vit_alpha_sweep/checkpoints  (local primary)


## 3. Config

In [5]:
TILES = {
    'polk_iowa':      'Polk County, Iowa (row-crop)',
    'lancaster_pa':   'Lancaster, Pennsylvania (mixed ag)',
    'hartford_ct':    'Hartford, Connecticut (urban-forest)',
    'everglades_fl':  'Everglades, Florida (wetland)',
    'lubbock_tx':     'Lubbock, Texas (irrigated dryland)',
    'sacramento_ca':  'Sacramento Delta, California',
    'phoenix_az':     'Phoenix, Arizona (desert-urban)',
    'yellowstone_wy': 'Yellowstone, Wyoming (forest)',
    'seattle_wa':     'Seattle, Washington (urban-water)',
    'mississippi_la': 'Mississippi Delta, Louisiana',
}
SIZES        = [100, 200, 500]
RADIUS_GRID  = [1, 2, 3, 5, 10]
BW_GRID      = [3, 5, 10, 20, 50]
ALPHA_GRID   = (0.05, 0.10, 0.15)
ALPHA_CV     = 0.10
MAX_CAL      = 20000
CV_FOLDS     = 5
LMD          = 0.5
SEED         = 0

# ViT hyperparameters
PATCH_SIZE   = 16           # spatial context window around each labeled pixel
SUB_PATCH    = 4            # ViT sub-patch (16/4 = 4 -> 16 tokens + 1 CLS)
VIT_DIM      = 96
VIT_DEPTH    = 6
VIT_HEADS    = 8
VIT_FFN      = 256
VIT_DROP     = 0.1
EPOCHS       = 50
LR           = 3e-4
BATCH_TR     = 64
BATCH_INF    = 256
print(f'{len(TILES)} tiles x {len(SIZES)} sizes; ViT {PATCH_SIZE}x{PATCH_SIZE} patch, '
      f'sub-patch {SUB_PATCH}, dim {VIT_DIM}, depth {VIT_DEPTH}')

10 tiles x 3 sizes; ViT 16x16 patch, sub-patch 4, dim 96, depth 6


## 4. Helpers — ViT + APS + SACP + GeoCP

In [6]:
import torch.nn as nn, torch.nn.functional as F, torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split, KFold
from scipy.spatial.distance import cdist
from scipy.signal import convolve2d
from scipy import stats as sstats
from collections import Counter

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', device)

# ---------- Vision Transformer for S2 patches ----------
class S2ViT(nn.Module):
    """Vision Transformer adapted for multispectral patch classification.
    Reference: Dosovitskiy et al. 2021. Input adapted to 13-band S2 patches."""
    def __init__(self, n_bands=13, n_classes=2, patch_size=PATCH_SIZE,
                 sub_patch=SUB_PATCH, dim=VIT_DIM, depth=VIT_DEPTH,
                 heads=VIT_HEADS, ffn=VIT_FFN, dropout=VIT_DROP):
        super().__init__()
        assert patch_size % sub_patch == 0
        self.patch_size = patch_size; self.sub_patch = sub_patch
        self.n_grid = patch_size // sub_patch          # 16/4 = 4
        self.n_tokens = self.n_grid * self.n_grid       # 16
        # Patch embedding: Conv2d with kernel=sub_patch, stride=sub_patch
        self.patch_embed = nn.Conv2d(n_bands, dim, kernel_size=sub_patch, stride=sub_patch)
        self.cls_token = nn.Parameter(torch.randn(1, 1, dim) * 0.02)
        self.pos_embed = nn.Parameter(torch.randn(1, self.n_tokens + 1, dim) * 0.02)
        enc_layer = nn.TransformerEncoderLayer(d_model=dim, nhead=heads, dim_feedforward=ffn,
                                               dropout=dropout, batch_first=True,
                                               activation='gelu', norm_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=depth)
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, n_classes)

    def forward(self, x):
        # x: (B, C, H, W) with H=W=patch_size, C=n_bands
        x = self.patch_embed(x)                        # (B, dim, n_grid, n_grid)
        x = x.flatten(2).transpose(1, 2)               # (B, n_tokens, dim)
        cls = self.cls_token.expand(x.size(0), -1, -1)
        x = torch.cat([cls, x], dim=1) + self.pos_embed
        x = self.encoder(x)
        x = self.norm(x[:, 0])                         # CLS token
        return self.head(x)

# ---------- Patch extraction from S2 emb (pad with reflect) ----------
def extract_s2_patches(emb_chw, indices, W, patch_size=PATCH_SIZE):
    """emb_chw: (C, H, W) float32 (we'll convert from emb (H, W, C)).
    indices: flat indices into H*W grid.
    Returns: (N, C, patch_size, patch_size) float32."""
    C, Hh, Ww = emb_chw.shape
    half = patch_size // 2
    padded = np.pad(emb_chw, ((0,0),(half, half-1),(half, half-1)), mode='reflect')
    out = np.zeros((len(indices), C, patch_size, patch_size), dtype=np.float32)
    for e, idx in enumerate(indices):
        r, c = idx // W, idx % W
        out[e] = padded[:, r:r+patch_size, c:c+patch_size]
    return out

# ---------- CP helpers ----------
def aps_scores(probs, labels=None, rng=None):
    rng = rng or np.random
    n, K = probs.shape
    si = np.argsort(-probs, axis=1)
    sp = np.take_along_axis(probs, si, axis=1)
    cs = np.cumsum(sp, axis=1)
    U = rng.uniform(0, 1, size=(n, K))
    ss = cs - sp * U
    scores = np.zeros_like(ss)
    for i in range(n):
        scores[i, si[i]] = ss[i]
    if labels is not None:
        return np.array([scores[i, int(labels[i])] for i in range(n)])
    return scores

def conformal_quantile(scores, alpha):
    n = len(scores)
    return float(np.quantile(scores, np.ceil((n+1)*(1-alpha))/n))

def set_interval_score_vec(score_mat, q_vec, y_true, alpha):
    in_set = score_mat < q_vec[:, None]
    sizes  = in_set.sum(axis=1)
    covered = in_set[np.arange(len(y_true)), y_true.astype(int)]
    return float((sizes + (2.0/alpha)*(~covered).astype(np.float64)).mean())

def eval_method(score_mat, q_scalar_or_vec, y_true, alpha):
    q_vec = np.full(len(y_true), q_scalar_or_vec) if np.isscalar(q_scalar_or_vec) else q_scalar_or_vec
    in_set = score_mat < q_vec[:, None]
    sizes  = in_set.sum(axis=1)
    covered = in_set[np.arange(len(y_true)), y_true.astype(int)]
    IS = float((sizes + (2.0/alpha)*(~covered).astype(np.float64)).mean())
    buckets = [(1,1),(2,2),(3,3),(4,4),(5,10**9)]
    worst = 0.0; rows = []
    for lo, hi in buckets:
        m = (sizes >= lo) & (sizes <= hi)
        if m.sum() == 0: rows.append((lo, hi, 0, None)); continue
        c = float(covered[m].mean()); rows.append((lo, hi, int(m.sum()), c))
        worst = max(worst, abs(c - (1 - alpha)))
    return dict(is_=IS, cov=float(covered.mean()), size=float(sizes.mean()),
                sscv_pct=worst*100, sscv_per_bucket=rows,
                in_set=in_set, sizes=sizes.astype(np.int32), covered=covered.astype(bool))

def sacp_smooth_radius(score_map, H, W, valid_idx, radius, lmd=0.5, k_iter=1):
    N, K = score_map.shape
    if radius <= 0: return score_map.copy()
    mask = np.zeros(N, dtype=bool); mask[valid_idx] = True
    mask_2d = mask.reshape(H, W).astype(np.float64)
    score_2d = score_map.reshape(H, W, K).astype(np.float64)
    k_size = 2 * radius + 1
    kernel = np.ones((k_size, k_size), dtype=np.float64); kernel[radius, radius] = 0.0
    for _ in range(k_iter):
        den = convolve2d(mask_2d, kernel, mode='same', boundary='fill', fillvalue=0.0)
        smoothed = np.empty_like(score_2d)
        for cls in range(K):
            values = np.where(mask_2d > 0, score_2d[..., cls], 0.0)
            num = convolve2d(values, kernel, mode='same', boundary='fill', fillvalue=0.0)
            smoothed[..., cls] = np.where(den > 1e-10, num / np.maximum(den, 1e-10), 0.0)
        new_score = (1 - lmd) * score_2d + lmd * smoothed
        score_2d = np.where(mask_2d[..., None] > 0, new_score, score_2d)
    return score_2d.reshape(N, K)

def vwq(sorted_scores, d_matrix, order, bw, alpha):
    log_w = -0.5 * (d_matrix / bw) ** 2
    log_w -= log_w.max(axis=1, keepdims=True)
    w = np.exp(log_w); w_sorted = w[:, order]
    ws = w_sorted / w_sorted.sum(axis=1, keepdims=True)
    cum = np.cumsum(ws, axis=1)
    k_star = np.argmax(cum >= (1 - alpha), axis=1)
    return sorted_scores[k_star]

def stratified_sub(labels, n, seed):
    if n is None or n >= len(labels): return np.arange(len(labels))
    uniq, cnts = np.unique(labels, return_counts=True)
    per_cls_n = np.maximum(1, (n * cnts / cnts.sum()).astype(int))
    rng = np.random.RandomState(seed); out = []
    for c, nc in zip(uniq, per_cls_n):
        pool = np.where(labels == c)[0]
        out.append(pool if len(pool) <= nc else rng.choice(pool, size=nc, replace=False))
    return np.concatenate(out)

print('Helpers + S2ViT defined.')

device = cuda
Helpers + S2ViT defined.


## 5. Per-(tile, scale) experiment

In [7]:
def run_tile_size(tile, size_px, alpha_grid=ALPHA_GRID, alpha_cv=ALPHA_CV,
                   radius_grid=RADIUS_GRID, bw_grid=BW_GRID,
                   max_cal=MAX_CAL, cv_folds=CV_FOLDS, lmd=LMD, seed=SEED):
    ckpt_path = f'{CKPT_DIR}/{tile}_s{size_px}.pkl'
    if os.path.exists(ckpt_path):
        with open(ckpt_path, 'rb') as f: return pickle.load(f)

    npz_path = f'{TILE_DIR}/{tile}.npz'
    arr = np.load(npz_path)
    emb_full, label_full = arr['emb'], arr['label']
    Hf, Wf = label_full.shape
    Seff = min(size_px, Hf, Wf)
    r0 = Hf // 2 - Seff // 2
    c0 = Wf // 2 - Seff // 2
    emb   = emb_full[r0:r0+Seff, c0:c0+Seff]
    label = label_full[r0:r0+Seff, c0:c0+Seff]

    H, W, D = emb.shape; N = H * W
    flat_label = label.ravel()
    labeled_idx = np.where(flat_label > 0)[0]
    y_raw = flat_label[labeled_idx]
    counts = Counter(y_raw.tolist())
    min_count = int(max(10, min(100, N // 500)))
    rare = [c for c, cnt in counts.items() if cnt < min_count]
    keep = ~np.isin(y_raw, rare)
    labeled_idx = labeled_idx[keep]; y_raw = y_raw[keep]
    classes = sorted(np.unique(y_raw).tolist()); K = len(classes)
    if K < 2 or len(y_raw) < 200:
        return {'tile': tile, 'size_px': Seff, 'degenerate': True, 'n_classes': K}
    cls_remap = {c: i for i, c in enumerate(classes)}
    y = np.array([cls_remap[v] for v in y_raw])

    idx_pos = np.arange(len(y))
    idx_tr, idx_tmp = train_test_split(idx_pos, train_size=0.6, random_state=seed*100+42, stratify=y)
    idx_ca, idx_te  = train_test_split(idx_tmp, test_size=0.5, random_state=seed*100+42, stratify=y[idx_tmp])

    # --- ViT training ---
    torch.manual_seed(seed*100+42); np.random.seed(seed*100+42)
    emb_chw = emb.transpose(2, 0, 1).astype(np.float32)
    tr_idx_flat = labeled_idx[idx_tr]; ca_idx_flat = labeled_idx[idx_ca]; te_idx_flat = labeled_idx[idx_te]
    X_tr = extract_s2_patches(emb_chw, tr_idx_flat, W)
    X_ca = extract_s2_patches(emb_chw, ca_idx_flat, W)
    X_te = extract_s2_patches(emb_chw, te_idx_flat, W)

    model = S2ViT(n_bands=D, n_classes=K).to(device)
    loader = DataLoader(TensorDataset(torch.FloatTensor(X_tr), torch.LongTensor(y[idx_tr])),
                        batch_size=BATCH_TR, shuffle=True)
    opt = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    crit = nn.CrossEntropyLoss()
    for _ in range(EPOCHS):
        model.train()
        for Xb, yb in loader:
            Xb, yb = Xb.to(device), yb.to(device)
            opt.zero_grad(); crit(model(Xb), yb).backward(); opt.step()
        sched.step()

    model.eval()
    def get_probs(X):
        loader = DataLoader(TensorDataset(torch.FloatTensor(X)), batch_size=BATCH_INF, shuffle=False)
        out = []
        with torch.no_grad():
            for (b,) in loader: out.append(torch.softmax(model(b.to(device)), dim=1).cpu().numpy())
        return np.concatenate(out)
    probs_ca = get_probs(X_ca); probs_te = get_probs(X_te)
    pred_te = np.argmax(probs_te, axis=1)
    acc = float((pred_te == y[idx_te]).mean())

    del X_tr, X_ca, X_te, model
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    rng = np.random.RandomState(seed*100+42)
    cal_all  = aps_scores(probs_ca, rng=rng)
    test_all = aps_scores(probs_te, rng=rng)
    cal_true = np.array([cal_all[i, int(y[idx_ca][i])] for i in range(len(idx_ca))])

    cal_flat_idx  = labeled_idx[idx_ca]
    test_flat_idx = labeled_idx[idx_te]
    score_map = np.zeros((N, K), dtype=np.float64)
    score_map[cal_flat_idx]  = cal_all
    score_map[test_flat_idx] = test_all
    valid_idx_all = np.concatenate([cal_flat_idx, test_flat_idx])
    fused_per_r = {r: sacp_smooth_radius(score_map, H, W, valid_idx_all, radius=r, lmd=lmd)
                   for r in radius_grid}
    fcu_per_r = {r: np.array([fused_per_r[r][cal_flat_idx[e], int(y[idx_ca][e])] for e in range(len(idx_ca))])
                 for r in radius_grid}
    ftu_per_r = {r: fused_per_r[r][test_flat_idx] for r in radius_grid}

    sub_ca = stratified_sub(y[idx_ca], max_cal, seed*100+43)
    n_sub = len(sub_ca)
    coords_sub = np.stack([cal_flat_idx[sub_ca] // W, cal_flat_idx[sub_ca] % W], 1).astype(float)
    y_sub = y[idx_ca][sub_ca]
    cal_sub_flat = cal_flat_idx[sub_ca]
    fcu_sub_per_r = {r: fcu_per_r[r][sub_ca] for r in radius_grid}

    bws_valid = [bw for bw in bw_grid if bw < Seff * 0.8]
    if not bws_valid: bws_valid = [max(2, Seff // 20)]

    kf = KFold(n_splits=cv_folds, shuffle=True, random_state=seed)
    cv_is_sacp  = {r: [] for r in radius_grid}
    cv_is_geocp = {(r, bw): [] for r in radius_grid for bw in bws_valid}
    for f_tr_idx, f_val_idx in kf.split(np.arange(n_sub)):
        y_cv_val = y_sub[f_val_idx]
        d_cv = cdist(coords_sub[f_val_idx], coords_sub[f_tr_idx])
        for r in radius_grid:
            fcu_tr = fcu_sub_per_r[r][f_tr_idx]
            fcu_val_all = fused_per_r[r][cal_sub_flat[f_val_idx]]
            q_fold = conformal_quantile(fcu_tr, alpha_cv)
            cv_is_sacp[r].append(
                set_interval_score_vec(fcu_val_all, np.full(len(f_val_idx), q_fold), y_cv_val, alpha_cv))
            order_tr = np.argsort(fcu_tr); sorted_tr = fcu_tr[order_tr]
            for bw in bws_valid:
                q_val = vwq(sorted_tr, d_cv, order_tr, bw, alpha_cv)
                cv_is_geocp[(r, bw)].append(
                    set_interval_score_vec(fcu_val_all, q_val, y_cv_val, alpha_cv))
    cv_sacp_mean  = {r: float(np.mean(v))  for r, v in cv_is_sacp.items()}
    cv_geocp_mean = {k: float(np.mean(v)) for k, v in cv_is_geocp.items()}
    best_r_sacp = int(min(cv_sacp_mean, key=cv_sacp_mean.get))
    best_rb     = min(cv_geocp_mean, key=cv_geocp_mean.get)
    best_r_geo, best_bw_geo = int(best_rb[0]), int(best_rb[1])

    fcu_C = fcu_per_r[best_r_geo][sub_ca]
    ftu_C = ftu_per_r[best_r_geo]
    order_C = np.argsort(fcu_C); sorted_C = fcu_C[order_C]
    coords_te = np.stack([test_flat_idx // W, test_flat_idx % W], 1).astype(float)
    n_te = len(test_flat_idx); batch_test = min(2000, max(200, n_te))
    q_C_per_alpha = {a: np.empty(n_te) for a in alpha_grid}
    for b in range(0, n_te, batch_test):
        be = min(b + batch_test, n_te)
        d = cdist(coords_te[b:be], coords_sub)
        for a in alpha_grid:
            q_C_per_alpha[a][b:be] = vwq(sorted_C, d, order_C, best_bw_geo, a)

    per_alpha = {}
    y_te_arr = y[idx_te]
    for a in alpha_grid:
        q_D_s = conformal_quantile(cal_true, a)
        q_A_s = conformal_quantile(fcu_per_r[1], a)
        q_B_s = conformal_quantile(fcu_per_r[best_r_sacp], a)
        blocks = {
            'D': eval_method(test_all,           q_D_s, y_te_arr, a),
            'A': eval_method(ftu_per_r[1],       q_A_s, y_te_arr, a),
            'B': eval_method(ftu_per_r[best_r_sacp], q_B_s, y_te_arr, a),
            'C': eval_method(ftu_C,              q_C_per_alpha[a], y_te_arr, a),
        }
        row = {}
        for lab, bk in blocks.items():
            row[lab] = {'cov': bk['cov'], 'size': bk['size'], 'is': bk['is_'],
                        'sscv_pct': bk['sscv_pct'], 'sscv_per_bucket': bk['sscv_per_bucket'],
                        'in_set': bk['in_set'].astype(bool),
                        'sizes':   bk['sizes'], 'covered': bk['covered']}
        row['q_C'] = q_C_per_alpha[a].astype(np.float32)
        per_alpha[f'{a:.2f}'] = row

    result = dict(
        tile=tile, backbone='ViT', size_px=int(Seff), size_km=Seff*0.01,
        H=int(H), W=int(W), n_classes=int(K),
        n_cal=int(len(idx_ca)), n_cal_sub=int(n_sub), n_test=int(len(idx_te)),
        alpha_grid=list(alpha_grid), alpha_cv=float(alpha_cv),
        accuracy=acc,
        best_r_sacp=best_r_sacp, best_r_geocp=best_r_geo, best_bw_geocp=best_bw_geo,
        per_alpha=per_alpha,
        cv_is_sacp=cv_sacp_mean,
        cv_is_geocp={f'{r}_{bw}': v for (r,bw), v in cv_geocp_mean.items()},
        probs_ca=probs_ca.astype(np.float32), probs_te=probs_te.astype(np.float32),
        cal_all_aps=cal_all.astype(np.float32), test_all_aps=test_all.astype(np.float32),
        cal_true_aps=cal_true.astype(np.float32),
        fcu_per_r={int(r): v.astype(np.float32) for r, v in fcu_per_r.items()},
        ftu_per_r={int(r): v.astype(np.float32) for r, v in ftu_per_r.items()},
        cal_flat_idx=cal_flat_idx.astype(np.int64),
        test_flat_idx=test_flat_idx.astype(np.int64),
        y_ca=y[idx_ca].astype(np.int64), y_te=y_te_arr.astype(np.int64),
        pred_te=pred_te.astype(np.int64),
        label=label.astype(np.int32),
    )
    tmp = ckpt_path + '.tmp'
    with open(tmp, 'wb') as f: pickle.dump(result, f, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(tmp, ckpt_path)
    return result

print('run_tile_size (ViT) defined')

run_tile_size (ViT) defined


## 6. Run all 30 configs (resumable)

In [ ]:
def _mirror(tile, size_px):
    src = f'{CKPT_DIR}/{tile}_s{size_px}.pkl'
    dst = f'{DRIVE_CKPT_DIR}/{tile}_s{size_px}.pkl'
    try:
        os.makedirs(DRIVE_CKPT_DIR, exist_ok=True); shutil.copy2(src, dst); return True
    except OSError: return False

total = len(TILES) * len(SIZES); done = 0; t_all = time.time()
log_file = f'{RESULTS_DIR}/run_log.txt'
for tile in TILES:
    print(f'\n{"="*80}\n{tile}  ({TILES[tile]})\n{"="*80}')
    for S in SIZES:
        ckpt = f'{CKPT_DIR}/{tile}_s{S}.pkl'
        if os.path.exists(ckpt):
            try:
                with open(ckpt, 'rb') as f: r = pickle.load(f)
            except OSError as e:
                print(f'  {S:>4d}px cached-read FAILED ({e}); will recompute.')
            else:
                done += 1
                if r.get('degenerate'):
                    print(f'  {S:>4d}px [cached, degenerate K={r["n_classes"]}]  [{done}/{total}]')
                else:
                    pa = r['per_alpha']
                    print(f'  {S:>4d}px [cached]  acc={r["accuracy"]:.3f}  '
                          f'α=0.10 IS: D={pa["0.10"]["D"]["is"]:.3f} A={pa["0.10"]["A"]["is"]:.3f} '
                          f'B={pa["0.10"]["B"]["is"]:.3f} C={pa["0.10"]["C"]["is"]:.3f}  '
                          f'(r*={r["best_r_geocp"]},bw*={r["best_bw_geocp"]})  [{done}/{total}]')
                continue
        t0 = time.time()
        try:
            r = run_tile_size(tile, S)
            done += 1
            mirrored = _mirror(tile, S)
            if r.get('degenerate'):
                msg = f'  {S:>4d}px DEGENERATE (K={r["n_classes"]})  [{time.time()-t0:.0f}s]  [{done}/{total}]'
            else:
                pa = r['per_alpha']
                msg = (f'  {S:>4d}px acc={r["accuracy"]:.3f}  '
                       f'α=0.10 IS: D={pa["0.10"]["D"]["is"]:.3f} A={pa["0.10"]["A"]["is"]:.3f} '
                       f'B={pa["0.10"]["B"]["is"]:.3f} C={pa["0.10"]["C"]["is"]:.3f}  '
                       f'(r*={r["best_r_geocp"]},bw*={r["best_bw_geocp"]})  '
                       f'[{time.time()-t0:.0f}s]  [{done}/{total}]  '
                       f'{"mirrored" if mirrored else "(mirror skipped)"}')
            print(msg)
            try:
                with open(log_file, 'a') as f: f.write(f'{tile} {msg}\n')
            except OSError: pass
        except Exception as e:
            import traceback; print(f'  {S:>4d}px FAILED: {e}'); traceback.print_exc()

print(f'\n{"="*80}\nALL DONE: {done}/{total} configs  ({(time.time()-t_all)/60:.1f} min)')


polk_iowa  (Polk County, Iowa (row-crop))
   100px [cached]  acc=0.861  α=0.10 IS: D=3.276 A=3.164 B=3.015 C=3.054  (r*=2,bw*=50)  [1/30]
   200px [cached]  acc=0.840  α=0.10 IS: D=3.373 A=3.262 B=3.229 C=3.257  (r*=5,bw*=10)  [2/30]


/tmp/ipykernel_5505/1034615945.py:31: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=depth)


## 7. Aggregate + paired tests

In [ ]:
import pandas as pd
rows = []
for tile in TILES:
    for S in SIZES:
        p = f'{CKPT_DIR}/{tile}_s{S}.pkl'
        if not os.path.exists(p): continue
        with open(p, 'rb') as f: r = pickle.load(f)
        if r.get('degenerate'): continue
        for a_str, block in r['per_alpha'].items():
            a = float(a_str)
            for lab in ('D','A','B','C'):
                b = block[lab]
                rows.append({'backbone': 'ViT', 'tile': tile, 'size_px': r['size_px'], 'size_km': r['size_km'],
                             'alpha': a, 'method': lab,
                             'is': b['is'], 'cov': b['cov'], 'size': b['size'], 'sscv_pct': b['sscv_pct'],
                             'accuracy': r['accuracy'],
                             'r_B': r['best_r_sacp'], 'r_C': r['best_r_geocp'], 'bw_C': r['best_bw_geocp']})
df = pd.DataFrame(rows)
df.to_csv(f'{RESULTS_DIR}/per_config_alpha_vit.csv', index=False)
print('wrote', f'{RESULTS_DIR}/per_config_alpha_vit.csv  ({len(df)} rows)')

print('\n=== Pooled α-sweep (S2-ViT) ===')
pooled = (df.groupby(['alpha','method']).agg(is_mean=('is','mean'), cov_mean=('cov','mean'),
    size_mean=('size','mean'), sscv_mean=('sscv_pct','mean'), n=('tile','count')).reset_index())
print(pooled.round(3).to_string(index=False))

print('\n=== Paired tests (C vs A, C vs D) ===')
for a in sorted(df.alpha.unique()):
    for ref in ('A','D'):
        sub_ref = df[(df.alpha==a)&(df.method==ref)].sort_values(['tile','size_px']).reset_index(drop=True)
        sub_C   = df[(df.alpha==a)&(df.method=='C')].sort_values(['tile','size_px']).reset_index(drop=True)
        impr = (sub_ref['is'].values - sub_C['is'].values) / sub_ref['is'].values * 100
        t, p = sstats.ttest_1samp(impr, 0.0)
        print(f'  α={a:.2f}  C vs {ref}: {impr.mean():+.2f}% ± {impr.std(ddof=1)/np.sqrt(len(impr)):.2f}  t={t:.2f}  p={p:.2e}  (n={len(impr)})')

## 8. Package

In [ ]:
zip_local = '/content/s2_vit_alpha_sweep_results.zip'
shutil.make_archive('/content/s2_vit_alpha_sweep_results', 'zip', LOCAL_DIR)
try:
    os.makedirs(DRIVE_MIRROR, exist_ok=True)
    shutil.copy2(zip_local, f'{DRIVE_MIRROR}/s2_vit_alpha_sweep_results.zip')
    print('Copied to', f'{DRIVE_MIRROR}/s2_vit_alpha_sweep_results.zip')
except OSError as e:
    print(f'Drive copy failed ({e}); use files.download("{zip_local}")')